Movie Rating Prediction (MovieLens 100K)

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

Load MovieLens 100K Dataset

In [2]:
import pandas as pd

columns = ["user_id", "movie_id", "rating", "timestamp"]

data = pd.read_csv(
    "u.data.zip",
    sep="\t",
    names=columns
)

data.head()


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


Create User–Movie Matrix

In [4]:
user_movie_matrix = data.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

user_movie_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Handle Missing Values

In [5]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

Compute User Similarity

In [6]:
user_similarity = cosine_similarity(user_movie_matrix_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
2,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
3,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
4,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
5,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


Rating Prediction Function

In [7]:
def predict_rating(user_id, movie_id):
    if movie_id not in user_movie_matrix.columns:
        return np.nan
    
    similarities = user_similarity_df[user_id]
    ratings = user_movie_matrix[movie_id].fillna(0)
    
    numerator = np.dot(similarities, ratings)
    denominator = np.sum(similarities)
    
    if denominator == 0:
        return np.nan
    
    return numerator / denominator

Model Evaluation (RMSE)

In [8]:
test_data = data.sample(1000, random_state=42)

predictions = []
actuals = []

for _, row in test_data.iterrows():
    pred = predict_rating(row["user_id"], row["movie_id"])
    if not np.isnan(pred):
        predictions.append(pred)
        actuals.append(row["rating"])

rmse = np.sqrt(mean_squared_error(actuals, predictions))
rmse

np.float64(2.7507604024332952)